In [240]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

In [241]:
df = pd.read_csv('../data/AI_Impact_Student_Life_2026.csv')
df.head()

,Student_ID,Age,Major,Primary_AI_Tool,Task_Frequency_Daily,Main_Usage_Case,GPA_Baseline,GPA_Post_AI,Time_Saved_Hours_Weekly,AI_Ethics_Concern,Career_Confidence_Score
0,STU-6019,23,Software Engineering,Gemini Pro,1,Code Debugging,2.62,2.62,9,Medium,3
1,STU-6962,22,Modern History,GitHub Copilot,3,Essay Drafting,3.99,4.00,7,Medium,4
2,STU-2338,18,Data Science,Perplexity,2,Literature Review,2.57,2.57,15,High,7
3,STU-1380,19,Biology,Claude 3.5,5,Essay Drafting,2.67,2.87,12,Low,5
4,STU-1837,19,Biology,ChatGPT-4o,10,Code Debugging,3.65,3.85,5,High,9


In [242]:
df.describe()

,Age,Task_Frequency_Daily,GPA_Baseline,GPA_Post_AI,Time_Saved_Hours_Weekly,Career_Confidence_Score
count,1500.000000,1500.000000,1500.000000,1500.000000,1500.00000,1500.000000
mean,21.494000,5.407333,3.256853,3.344587,8.51000,5.417333
std,2.297277,2.905462,0.430583,0.437409,4.07148,2.844977
min,18.000000,1.000000,2.500000,2.400000,2.00000,1.000000
25%,20.000000,3.000000,2.880000,2.980000,5.00000,3.000000
50%,21.000000,5.000000,3.260000,3.360000,9.00000,5.000000
75%,23.000000,8.000000,3.620000,3.710000,12.00000,8.000000
max,25.000000,10.000000,4.000000,4.000000,15.00000,10.000000


In [243]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Student_ID               1500 non-null   str    
 1   Age                      1500 non-null   int64  
 2   Major                    1500 non-null   str    
 3   Primary_AI_Tool          1500 non-null   str    
 4   Task_Frequency_Daily     1500 non-null   int64  
 5   Main_Usage_Case          1500 non-null   str    
 6   GPA_Baseline             1500 non-null   float64
 7   GPA_Post_AI              1500 non-null   float64
 8   Time_Saved_Hours_Weekly  1500 non-null   int64  
 9   AI_Ethics_Concern        1500 non-null   str    
 10  Career_Confidence_Score  1500 non-null   int64  
dtypes: float64(2), int64(4), str(5)
memory usage: 203.0 KB


In [244]:
new_df = df.drop(['Student_ID', 'Primary_AI_Tool'], axis=1)

In [245]:
new_df['Main_Usage_Case'].value_counts()

Main_Usage_Case
Exam Prep            341
Literature Review    308
Essay Drafting       301
Brainstorming        286
Code Debugging       264
Name: count, dtype: int64

In [246]:
new_df['Major'] = new_df['Major'].map({'Fine Arts': 1, 'Data Science': 2, 'Biology': 3, 'Business Administration': 4, 'Software Engineering': 5, 'Modern History': 6})
new_df['Main_Usage_Case'] = new_df['Main_Usage_Case'].map({'Exam Prep': 1, 'Literature Review': 2, 'Essay Drafting': 3, 'Brainstorming': 4, 'Code Debugging': 5})
new_df['AI_Ethics_Concern'] = new_df['AI_Ethics_Concern'].map({'Low': 1, 'Medium': 2, 'High': 3})

In [247]:
new_df['career confidence'] = '1'
new_df.loc[new_df['Career_Confidence_Score'] >=7, 'career confidence'] = '1'
new_df.loc[new_df['Career_Confidence_Score'] < 7, 'career confidence'] = '0'
new_df.head()

,Age,Major,Task_Frequency_Daily,Main_Usage_Case,GPA_Baseline,GPA_Post_AI,Time_Saved_Hours_Weekly,AI_Ethics_Concern,Career_Confidence_Score,career confidence
0,23,5,1,5,2.62,2.62,9,2,3,0
1,22,6,3,3,3.99,4.00,7,2,4,0
2,18,2,2,2,2.57,2.57,15,3,7,1
3,19,3,5,3,2.67,2.87,12,1,5,0
4,19,3,10,5,3.65,3.85,5,3,9,1


In [248]:
y = new_df['career confidence']
X = new_df.drop(['Career_Confidence_Score', 'career confidence'], axis=1)

In [249]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
import numpy as np

In [250]:
X_train, X_holdout, y_train, y_holdout = train_test_split(X, y, test_size=0.3, random_state=42)

In [251]:
tree = DecisionTreeClassifier()
np.mean(cross_val_score(tree, X_train, y_train, cv=5))

np.float64(0.5266666666666666)

In [252]:
tree.fit(X_train, y_train)
importance = pd.Series(tree.feature_importances_, index=X_train.columns)
importance.sort_values(ascending=False)

GPA_Baseline               0.221185
GPA_Post_AI                0.220305
Time_Saved_Hours_Weekly    0.158420
Task_Frequency_Daily       0.114129
Age                        0.112254
Main_Usage_Case            0.096425
Major                      0.040689
AI_Ethics_Concern          0.036594
dtype: float64

In [253]:
Knn = KNeighborsClassifier()
np.mean(cross_val_score(Knn, X_train, y_train, cv=5))

np.float64(0.5419047619047619)

In [254]:
tree_params = {'max_depth': np.arange(1, 15), 'max_features': [0.2, 0.4, 0.6, 0.8, 1]}
tree_grid = GridSearchCV(tree, tree_params, cv=5, n_jobs=-1)

In [255]:
tree_grid.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",DecisionTreeClassifier()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': array([ 1, 2..., 12, 13, 14]), 'max_features': [0.2, 0.4, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also dis

In [256]:
tree_grid.best_params_, tree_grid.best_score_

({'max_depth': np.int64(2), 'max_features': 0.2},
 np.float64(0.6295238095238095))

In [257]:
Knn_params = {'n_neighbors': np.arange(1, 30)}
Knn_grid = GridSearchCV(Knn, Knn_params, cv=5, n_jobs=-1)

In [258]:
Knn_grid.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",KNeighborsClassifier()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'n_neighbors': array([ 1, 2..., 27, 28, 29])}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : the fold and candid

In [259]:
Knn_grid.best_params_, Knn_grid.best_score_

({'n_neighbors': np.int64(18)}, np.float64(0.5980952380952382))

In [260]:
from sklearn.metrics import accuracy_score

In [261]:
tree_prediction = tree_grid.predict(X_holdout)
Knn_prediction = Knn_grid.predict(X_holdout)

In [262]:
accuracy_score(y_holdout, tree_prediction)

0.5977777777777777

In [263]:
accuracy_score(y_holdout, Knn_prediction)

0.5933333333333334